# Project FORESIGHT — Exploratory Data Analysis
### Demand & Inventory Intelligence for NorthBay Living

## 1. Business Context

NorthBay Living sells home & lifestyle products online and plans inventory manually. Two costly patterns show up when planning is done on gut feel: **stockouts** (best-sellers run out, sales are lost) and **overstock** (slow movers tie up cash and eventually get marked down).

This EDA explores the cleaned, joined `modeling_data.csv` (sales + product master + calendar + monthly inventory context, for the 50 SKUs that have both sales and master data) to answer:
- How is demand and revenue distributed across products, categories, and time?
- What seasonal or promotional patterns exist?
- Which products are financially risky (negative margin) or operationally risky (slow-moving, high inventory value)?
- What patterns should a demand-forecasting model account for later?

All numbers below are computed directly from the data — nothing here is assumed or fabricated.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

sns.set_style("whitegrid")
pd.options.display.float_format = '{:,.2f}'.format

DATA_PATH = Path("../data/processed/modeling_data.csv")
REPORTS_DIR = Path("../reports")
PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(DATA_PATH, parse_dates=["Date", "Launch_Date"])
df = df.sort_values(["SKU", "Date"]).reset_index(drop=True)

def fmt_inr(x):
    """Format a number as Indian Rupees for readable output."""
    return f"Rs {x:,.0f}"

df.head(3)

,Date,SKU,Units_Sold,Revenue,Price,Promotion,Product_Name,Category,Subcategory,Launch_Date,...,season,holiday,is_holiday,promotion_event,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,Inventory_Value
0,2024-01-01,SKU001,5,"18,320.25","3,664.05",0,Product 001,Furniture,Chair,2022-04-09,...,Winter,NaN,0,NaN,16,23,11,7,18,"58,420.96"
1,2024-01-02,SKU001,13,"47,632.65","3,664.05",0,Product 001,Furniture,Chair,2022-04-09,...,Winter,NaN,0,NaN,16,23,11,7,18,"58,420.96"
2,2024-01-03,SKU001,12,"43,968.60","3,664.05",0,Product 001,Furniture,Chair,2022-04-09,...,Winter,NaN,0,NaN,16,23,11,7,18,"58,420.96"


## 2. Data Overview

Quick structural check of the analysis-ready dataset before diving into patterns.

In [2]:
n_rows, n_cols = df.shape
date_min, date_max = df["Date"].min().date(), df["Date"].max().date()
n_skus = df["SKU"].nunique()
categories = sorted(df["Category"].unique())
subcategories = sorted(df["Subcategory"].unique())

print(f"Rows: {n_rows:,}   Columns: {n_cols}")
print(f"Date range: {date_min} to {date_max}  ({(date_max - date_min).days + 1} days)")
print(f"SKUs in this dataset: {n_skus}")
print(f"Categories ({len(categories)}): {categories}")
print(f"Subcategories ({len(subcategories)}): {subcategories}")
print()
print("Data-quality confirmation:")
print(f"  Missing values in key columns: {df[['Units_Sold','Revenue','Price','Category','Current_Stock']].isna().sum().sum()}")
print(f"  Duplicate (Date, SKU) rows: {df.duplicated(subset=['Date','SKU']).sum()}")
print("  Note: inventory_snapshots.csv contains 200 SKUs, but only 50 have matching sales/master")
print("  data. This EDA and future modeling use only those 50 documented, matched SKUs")
print("  (see reports/data_quality_report.txt for the full mismatch finding).")

df[["Units_Sold","Revenue","Price","Cost_Price","Selling_Price","Gross_Margin_Per_Unit"]].describe().T

Rows: 36,550   Columns: 29
Date range: 2024-01-01 to 2025-12-31  (731 days)
SKUs in this dataset: 50
Categories (5): ['Furniture', 'Home Decor', 'Kitchen', 'Lighting', 'Storage']
Subcategories (10): ['Cabinet', 'Chair', 'Cookware', 'Cushion', 'Lamp', 'Organizer', 'Rug', 'Shelf', 'Sofa', 'Table']

Data-quality confirmation:
  Missing values in key columns: 0
  Duplicate (Date, SKU) rows: 0
  Note: inventory_snapshots.csv contains 200 SKUs, but only 50 have matching sales/master
  data. This EDA and future modeling use only those 50 documented, matched SKUs
  (see reports/data_quality_report.txt for the full mismatch finding).


,count,mean,std,min,25%,50%,75%,max
Units_Sold,"36,550.00",14.00,8.47,0.00,7.00,13.00,20.00,56.00
Revenue,"36,550.00","84,707.43","76,534.63",0.00,"27,643.70","56,125.90","127,954.32","546,977.00"
Price,"36,550.00","5,892.66","3,138.78",663.46,"3,484.09","5,649.80","8,516.60","11,642.45"
Cost_Price,"36,550.00","3,770.73","2,005.45",307.06,"1,913.03","3,836.85","5,539.34","7,748.20"
Selling_Price,"36,550.00","5,892.66","3,138.78",663.46,"3,484.09","5,649.80","8,516.60","11,642.45"
Gross_Margin_Per_Unit,"36,550.00","2,121.93","3,746.90","-5,484.32",-681.48,"1,790.03","4,923.44","10,224.93"


## 3. Overall Sales Performance

Headline demand and revenue figures, then how they trend over time.

In [3]:
total_units = df["Units_Sold"].sum()
total_revenue = df["Revenue"].sum()
avg_daily_units = df.groupby("Date")["Units_Sold"].sum().mean()
n_days = df["Date"].nunique()

print(f"Total units sold ({n_days} days, {n_skus} SKUs): {total_units:,.0f}")
print(f"Total revenue: {fmt_inr(total_revenue)}")
print(f"Average units sold per day (across all SKUs): {avg_daily_units:,.1f}")
print(f"Average revenue per day: {fmt_inr(total_revenue / n_days)}")

Total units sold (731 days, 50 SKUs): 511,810
Total revenue: Rs 3,096,056,707
Average units sold per day (across all SKUs): 700.2
Average revenue per day: Rs 4,235,372


In [4]:
daily = df.groupby("Date").agg(Units_Sold=("Units_Sold","sum"), Revenue=("Revenue","sum")).reset_index()
daily["Revenue_7d_avg"] = daily["Revenue"].rolling(7, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily["Date"], y=daily["Revenue"], mode="lines",
                          name="Daily revenue", line=dict(color="lightsteelblue", width=1)))
fig.add_trace(go.Scatter(x=daily["Date"], y=daily["Revenue_7d_avg"], mode="lines",
                          name="7-day average", line=dict(color="darkblue", width=2)))
fig.update_layout(title="Daily Revenue Trend (with 7-day rolling average)",
                   xaxis_title="Date", yaxis_title="Revenue (Rs)",
                   template="plotly_white", height=420)
fig.show()

In [5]:
monthly = df.groupby(pd.Grouper(key="Date", freq="MS")).agg(
    Units_Sold=("Units_Sold","sum"), Revenue=("Revenue","sum")).reset_index()

fig = px.bar(monthly, x="Date", y="Units_Sold", title="Monthly Demand (Units Sold)",
             labels={"Units_Sold":"Units Sold", "Date":"Month"}, template="plotly_white")
fig.update_layout(height=400)
fig.show()

print(f"Highest-demand month: {monthly.loc[monthly['Units_Sold'].idxmax(),'Date'].strftime('%B %Y')} "
      f"({monthly['Units_Sold'].max():,.0f} units)")
print(f"Lowest-demand month:  {monthly.loc[monthly['Units_Sold'].idxmin(),'Date'].strftime('%B %Y')} "
      f"({monthly['Units_Sold'].min():,.0f} units)")

Highest-demand month: March 2024 (26,791 units)
Lowest-demand month:  October 2024 (17,173 units)


## 4. SKU Performance

Which products drive the business, and which are barely moving.

In [6]:
sku_perf = df.groupby(["SKU","Product_Name"]).agg(
    Units_Sold=("Units_Sold","sum"), Revenue=("Revenue","sum")).reset_index()
sku_perf["Revenue_Share_%"] = 100 * sku_perf["Revenue"] / sku_perf["Revenue"].sum()

top10_units = sku_perf.sort_values("Units_Sold", ascending=False).head(10)
top10_revenue = sku_perf.sort_values("Revenue", ascending=False).head(10)
bottom10_units = sku_perf.sort_values("Units_Sold", ascending=True).head(10)

fig = px.bar(top10_units, x="Units_Sold", y="Product_Name", orientation="h",
             title="Top 10 SKUs by Units Sold", template="plotly_white",
             labels={"Units_Sold":"Units Sold","Product_Name":""})
fig.update_layout(yaxis=dict(categoryorder="total ascending"), height=450)
fig.show()

In [7]:
fig = px.bar(top10_revenue, x="Revenue", y="Product_Name", orientation="h",
             title="Top 10 SKUs by Revenue", template="plotly_white",
             labels={"Revenue":"Revenue (Rs)","Product_Name":""})
fig.update_layout(yaxis=dict(categoryorder="total ascending"), height=450)
fig.show()

print("Slowest-moving 10 SKUs by total units sold over the full period:")
bottom10_units[["SKU","Product_Name","Units_Sold","Revenue"]]

Slowest-moving 10 SKUs by total units sold over the full period:


,SKU,Product_Name,Units_Sold,Revenue
10,SKU011,Product 011,1951,"2,818,336.56"
24,SKU025,Product 025,1976,"22,838,627.76"
38,SKU039,Product 039,2300,"16,373,585.00"
14,SKU015,Product 015,2994,"18,599,087.28"
3,SKU004,Product 004,3380,"23,192,106.60"
29,SKU030,Product 030,3493,"32,951,250.43"
35,SKU036,Product 036,3508,"19,949,996.00"
27,SKU028,Product 028,4101,"14,288,253.09"
2,SKU003,Product 003,4214,"34,041,661.22"
49,SKU050,Product 050,4537,"4,004,900.64"


In [8]:
# Pareto view: revenue concentration across SKUs
sku_sorted = sku_perf.sort_values("Revenue", ascending=False).reset_index(drop=True)
sku_sorted["Cumulative_Revenue_%"] = 100 * sku_sorted["Revenue"].cumsum() / sku_sorted["Revenue"].sum()

fig = go.Figure()
fig.add_trace(go.Bar(x=sku_sorted["SKU"], y=sku_sorted["Revenue"], name="Revenue"))
fig.add_trace(go.Scatter(x=sku_sorted["SKU"], y=sku_sorted["Cumulative_Revenue_%"],
                          name="Cumulative %", yaxis="y2", line=dict(color="firebrick")))
fig.update_layout(title="Revenue Pareto Across 50 SKUs",
                   yaxis=dict(title="Revenue (Rs)"),
                   yaxis2=dict(title="Cumulative % of Revenue", overlaying="y", side="right", range=[0,105]),
                   template="plotly_white", height=450, xaxis=dict(showticklabels=False))
fig.show()

n_skus_for_80pct = (sku_sorted["Cumulative_Revenue_%"] <= 80).sum() + 1
print(f"{n_skus_for_80pct} of {len(sku_sorted)} SKUs ({100*n_skus_for_80pct/len(sku_sorted):.0f}%) "
      f"generate about 80% of total revenue.")

24 of 50 SKUs (48%) generate about 80% of total revenue.


## 5. Category & Subcategory Analysis

In [9]:
cat_perf = df.groupby("Category").agg(
    Revenue=("Revenue","sum"), Units_Sold=("Units_Sold","sum")).reset_index()
cat_perf["Avg_Daily_Units"] = cat_perf["Units_Sold"] / n_days
cat_perf = cat_perf.sort_values("Revenue", ascending=False)

fig = px.bar(cat_perf, x="Category", y="Revenue", title="Revenue by Category",
             template="plotly_white", labels={"Revenue":"Revenue (Rs)"})
fig.show()

fig = px.bar(cat_perf, x="Category", y="Units_Sold", title="Units Sold by Category",
             template="plotly_white", labels={"Units_Sold":"Units Sold"})
fig.show()

strongest_cat = cat_perf.iloc[0]["Category"]
weakest_cat = cat_perf.iloc[-1]["Category"]
print(f"Strongest category by revenue: {strongest_cat} ({fmt_inr(cat_perf.iloc[0]['Revenue'])})")
print(f"Weakest category by revenue:   {weakest_cat} ({fmt_inr(cat_perf.iloc[-1]['Revenue'])})")

cat_perf

Strongest category by revenue: Home Decor (Rs 888,468,078)
Weakest category by revenue:   Lighting (Rs 458,736,187)


,Category,Revenue,Units_Sold,Avg_Daily_Units
1,Home Decor,"888,468,077.72",132570,181.35
0,Furniture,"603,390,468.06",82298,112.58
4,Storage,"581,927,421.83",93367,127.73
2,Kitchen,"563,534,552.46",101144,138.36
3,Lighting,"458,736,187.15",102431,140.12


In [10]:
subcat_perf = df.groupby(["Category","Subcategory"]).agg(
    Avg_Units_Per_Day=("Units_Sold", lambda s: s.sum() / n_days),
    Revenue=("Revenue","sum")).reset_index().sort_values("Revenue", ascending=False)
subcat_perf

,Category,Subcategory,Avg_Units_Per_Day,Revenue
8,Storage,Lamp,70.23,"469,259,152.30"
2,Home Decor,Cabinet,102.92,"467,618,260.35"
3,Home Decor,Table,78.44,"420,849,817.37"
7,Lighting,Organizer,70.91,"321,766,548.76"
1,Furniture,Shelf,57.09,"310,122,678.54"
4,Kitchen,Cushion,75.01,"298,023,898.36"
0,Furniture,Chair,55.49,"293,267,789.52"
5,Kitchen,Rug,63.35,"265,510,654.10"
6,Lighting,Cookware,69.21,"136,969,638.39"
9,Storage,Sofa,57.49,"112,668,269.53"


## 6. Seasonality & Time Patterns

In [11]:
season_perf = df.groupby("season")["Units_Sold"].sum().reindex(
    ["Winter","Spring","Summer","Monsoon","Autumn"]).dropna()

fig = px.bar(season_perf, x=season_perf.index, y=season_perf.values,
             title="Total Demand by Season", labels={"x":"Season","y":"Units Sold"},
             template="plotly_white")
fig.show()

dow_perf = df.groupby("day_of_week")["Units_Sold"].mean().reindex(
    ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"])

fig = px.bar(dow_perf, x=dow_perf.index, y=dow_perf.values,
             title="Average Daily Units Sold by Day of Week",
             labels={"x":"Day","y":"Avg Units Sold (all SKUs combined)"}, template="plotly_white")
fig.show()

holiday_avg = df.groupby("is_holiday")["Units_Sold"].mean()
print(f"Average units sold per SKU-day on holidays:     {holiday_avg.get(1, float('nan')):.2f}")
print(f"Average units sold per SKU-day on non-holidays: {holiday_avg.get(0, float('nan')):.2f}")
print(f"Peak season: {season_perf.idxmax()}  |  Lowest season: {season_perf.idxmin()}")
print(f"Peak day of week (avg demand): {dow_perf.idxmax()}  |  Lowest: {dow_perf.idxmin()}")

Average units sold per SKU-day on holidays:     12.16
Average units sold per SKU-day on non-holidays: 14.02
Peak season: Winter  |  Lowest season: Autumn
Peak day of week (avg demand): Sunday  |  Lowest: Tuesday


## 7. Promotion Analysis

Comparing promotional vs non-promotional days. This is a descriptive comparison of observed
averages — it does not establish that promotions *cause* the difference (other factors like
season or day-of-week could coincide with promotions).

In [12]:
promo_compare = df.groupby("Promotion").agg(
    Avg_Units=("Units_Sold","mean"), Avg_Revenue=("Revenue","mean"), Rows=("Units_Sold","size")).rename(
    index={0:"Non-Promotion", 1:"Promotion"})
promo_compare

,Avg_Units,Avg_Revenue,Rows
Promotion,,,
Non-Promotion,13.48,"81,550.08",32800
Promotion,18.61,"112,323.77",3750


In [13]:
fig = px.bar(promo_compare.reset_index(), x="Promotion", y="Avg_Units",
             title="Average Units Sold: Promotion vs Non-Promotion Days",
             labels={"Avg_Units":"Avg Units Sold per SKU-day"}, template="plotly_white")
fig.show()

lift_units = 100 * (promo_compare.loc["Promotion","Avg_Units"] / promo_compare.loc["Non-Promotion","Avg_Units"] - 1)
print(f"Observed difference in average units sold on promotion days vs non-promotion days: {lift_units:+.1f}%")

promo_by_cat = df.groupby(["Category","Promotion"])["Units_Sold"].mean().unstack().rename(
    columns={0:"Non-Promotion", 1:"Promotion"})
promo_by_cat

Observed difference in average units sold on promotion days vs non-promotion days: +38.1%


Promotion,Non-Promotion,Promotion
Category,,
Furniture,10.84,14.96
Home Decor,17.46,24.06
Kitchen,13.32,18.32
Lighting,13.48,18.69
Storage,12.28,17.04


## 8. Product / Margin Analysis

In [14]:
margin_by_sku = df.groupby(["SKU","Product_Name"]).agg(
    Gross_Margin_Per_Unit=("Gross_Margin_Per_Unit","first"),
    Units_Sold=("Units_Sold","sum")).reset_index()

n_negative_margin = (margin_by_sku["Gross_Margin_Per_Unit"] < 0).sum()
pct_negative_margin = 100 * n_negative_margin / len(margin_by_sku)
print(f"SKUs with negative gross margin: {n_negative_margin} of {len(margin_by_sku)} ({pct_negative_margin:.0f}%)")

margin_sorted = margin_by_sku.sort_values("Gross_Margin_Per_Unit")
fig = px.bar(pd.concat([margin_sorted.head(10), margin_sorted.tail(10)]),
             x="Gross_Margin_Per_Unit", y="Product_Name", orientation="h",
             title="Lowest and Highest Gross-Margin-Per-Unit Products",
             color="Gross_Margin_Per_Unit", color_continuous_scale="RdYlGn",
             template="plotly_white")
fig.update_layout(yaxis=dict(categoryorder="total ascending"), height=550)
fig.show()

SKUs with negative gross margin: 16 of 50 (32%)


In [15]:
fig = px.scatter(margin_by_sku, x="Units_Sold", y="Gross_Margin_Per_Unit",
                  hover_name="Product_Name",
                  title="Total Units Sold vs Gross Margin Per Unit (each point = one SKU)",
                  labels={"Units_Sold":"Total Units Sold","Gross_Margin_Per_Unit":"Gross Margin per Unit (Rs)"},
                  template="plotly_white")
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

corr = margin_by_sku["Units_Sold"].corr(margin_by_sku["Gross_Margin_Per_Unit"])
print(f"Correlation between total units sold and gross margin per unit: {corr:.2f}")
print("A correlation near zero indicates high-selling products are not necessarily high-margin ones.")

Correlation between total units sold and gross margin per unit: 0.14
A correlation near zero indicates high-selling products are not necessarily high-margin ones.


## 9. Inventory / Slow-Moving Analysis

Inventory snapshots are monthly per SKU. To avoid treating a repeated daily value as an
independent daily measurement, the figures below use one row per (SKU, month) rather than
the daily-joined table.

In [16]:
inv_monthly = df.drop_duplicates(subset=["SKU","year","month"])[
    ["SKU","Product_Name","year","month","Current_Stock","On_Order","Safety_Stock",
     "Reorder_Point","Inventory_Value"]].copy()

# Latest available monthly snapshot per SKU
latest_snapshot = inv_monthly.sort_values(["year","month"]).groupby("SKU").tail(1)

# Average daily demand per SKU over the full history, for a simple stock-vs-demand view
avg_daily_demand = df.groupby("SKU")["Units_Sold"].mean().rename("Avg_Daily_Demand")
latest_snapshot = latest_snapshot.merge(avg_daily_demand, on="SKU")
latest_snapshot["Days_of_Stock_On_Hand"] = (
    latest_snapshot["Current_Stock"] / latest_snapshot["Avg_Daily_Demand"].replace(0, np.nan))

top_inv_value = latest_snapshot.sort_values("Inventory_Value", ascending=False).head(10)
fig = px.bar(top_inv_value, x="Inventory_Value", y="Product_Name", orientation="h",
             title="Top 10 SKUs by Latest Inventory Value",
             labels={"Inventory_Value":"Inventory Value (Rs)"}, template="plotly_white")
fig.update_layout(yaxis=dict(categoryorder="total ascending"), height=450)
fig.show()

In [17]:
# Apparent slow movers: high days-of-stock relative to demand (top quartile)
threshold = latest_snapshot["Days_of_Stock_On_Hand"].quantile(0.75)
slow_movers = latest_snapshot[latest_snapshot["Days_of_Stock_On_Hand"] >= threshold].sort_values(
    "Days_of_Stock_On_Hand", ascending=False)

print(f"{len(slow_movers)} SKUs have {threshold:.0f}+ days of stock on hand relative to their "
      f"average daily demand (based on the latest monthly snapshot) — apparent slow movers to review:")
slow_movers[["SKU","Product_Name","Current_Stock","Avg_Daily_Demand","Days_of_Stock_On_Hand","Inventory_Value"]].head(10)

13 SKUs have 34+ days of stock on hand relative to their average daily demand (based on the latest monthly snapshot) — apparent slow movers to review:


,SKU,Product_Name,Current_Stock,Avg_Daily_Demand,Days_of_Stock_On_Hand,Inventory_Value
24,SKU025,Product 025,1124,2.70,415.81,"4,384,724.00"
10,SKU011,Product 011,753,2.67,282.13,"2,911,918.77"
35,SKU036,Product 036,383,4.80,79.81,"574,446.38"
19,SKU020,Product 020,1002,13.87,72.25,"5,550,418.68"
2,SKU003,Product 003,349,5.76,60.54,"2,145,941.67"
3,SKU004,Product 004,259,4.62,56.01,"321,905.92"
5,SKU006,Product 006,359,6.55,54.83,"2,378,877.60"
0,SKU001,Product 001,771,14.37,53.67,"2,815,160.01"
49,SKU050,Product 050,328,6.21,52.85,"1,138,983.28"
1,SKU002,Product 002,526,10.14,51.88,"2,974,319.60"


## 10. Demand Volatility

Volatility matters for forecasting: a highly variable SKU is harder to predict accurately and
usually needs more safety stock to avoid stockouts, while a stable SKU can be forecast with
tighter confidence.

In [18]:
vol = df.groupby(["SKU","Product_Name"])["Units_Sold"].agg(["mean","std"]).reset_index()
vol["Coefficient_of_Variation"] = vol["std"] / vol["mean"]
vol_sorted = vol.sort_values("Coefficient_of_Variation", ascending=False)

fig = px.histogram(vol, x="Coefficient_of_Variation", nbins=20,
                    title="Distribution of Demand Volatility (Coefficient of Variation) Across SKUs",
                    template="plotly_white")
fig.show()

print("Most volatile SKUs (hardest to forecast reliably):")
display_cols = ["SKU","Product_Name","mean","std","Coefficient_of_Variation"]
print(vol_sorted[display_cols].head(5).to_string(index=False))
print()
print("Most stable SKUs (easiest to forecast reliably):")
print(vol_sorted[display_cols].tail(5).to_string(index=False))

Most volatile SKUs (hardest to forecast reliably):
   SKU Product_Name  mean  std  Coefficient_of_Variation
SKU011  Product 011  2.67 1.74                      0.65
SKU025  Product 025  2.70 1.70                      0.63
SKU039  Product 039  3.15 1.83                      0.58
SKU015  Product 015  4.10 2.20                      0.54
SKU004  Product 004  4.62 2.36                      0.51

Most stable SKUs (easiest to forecast reliably):
   SKU Product_Name  mean  std  Coefficient_of_Variation
SKU007  Product 007 24.80 6.53                      0.26
SKU027  Product 027 24.24 6.32                      0.26
SKU037  Product 037 24.84 6.43                      0.26
SKU018  Product 018 25.24 6.53                      0.26
SKU049  Product 049 25.13 6.29                      0.25


## 11. Business Insights

Findings below are pulled directly from the computed values above.

In [19]:
insights = [
    f"1. {strongest_cat} is the strongest category by revenue ({fmt_inr(cat_perf.iloc[0]['Revenue'])}), "
    f"while {weakest_cat} is the weakest ({fmt_inr(cat_perf.iloc[-1]['Revenue'])}).",

    f"2. Revenue is concentrated: just {n_skus_for_80pct} of {len(sku_sorted)} SKUs "
    f"(~{100*n_skus_for_80pct/len(sku_sorted):.0f}%) account for roughly 80% of total revenue.",

    f"3. {monthly.loc[monthly['Units_Sold'].idxmax(),'Date'].strftime('%B %Y')} was the peak demand month "
    f"({monthly['Units_Sold'].max():,.0f} units); {season_perf.idxmax()} is the strongest season overall.",

    f"4. On days flagged as promotions, average units sold per SKU-day were {lift_units:+.1f}% versus "
    f"non-promotion days — a notable observed difference, though not proof of causation.",

    f"5. {n_negative_margin} of {len(margin_by_sku)} SKUs ({pct_negative_margin:.0f}%) currently have a "
    f"negative gross margin per unit — these are losing money on every sale.",

    f"6. Total units sold and gross margin per unit have a correlation of {corr:.2f} across SKUs, "
    f"showing high sales volume does not reliably mean high profitability.",

    f"7. {len(slow_movers)} SKUs show {threshold:.0f}+ days of stock on hand relative to their demand in the "
    f"latest snapshot — capital tied up in slow-moving inventory.",

    f"8. Demand volatility (coefficient of variation) ranges from {vol['Coefficient_of_Variation'].min():.2f} "
    f"to {vol['Coefficient_of_Variation'].max():.2f} across SKUs, meaning a one-size-fits-all forecasting "
    f"approach will not suit every product equally well.",
]

for line in insights:
    print(line)
    print()

1. Home Decor is the strongest category by revenue (Rs 888,468,078), while Lighting is the weakest (Rs 458,736,187).

2. Revenue is concentrated: just 24 of 50 SKUs (~48%) account for roughly 80% of total revenue.

3. March 2024 was the peak demand month (26,791 units); Winter is the strongest season overall.

4. On days flagged as promotions, average units sold per SKU-day were +38.1% versus non-promotion days — a notable observed difference, though not proof of causation.

5. 16 of 50 SKUs (32%) currently have a negative gross margin per unit — these are losing money on every sale.

6. Total units sold and gross margin per unit have a correlation of 0.14 across SKUs, showing high sales volume does not reliably mean high profitability.

7. 13 SKUs show 34+ days of stock on hand relative to their demand in the latest snapshot — capital tied up in slow-moving inventory.

8. Demand volatility (coefficient of variation) ranges from 0.25 to 0.65 across SKUs, meaning a one-size-fits-all for

## 12. Forecasting Preparation

Patterns observed above that a demand-forecasting model (built in a later notebook) should account for:

- **Trend** — the monthly demand series shows month-to-month movement rather than a flat line, so a naive constant-mean forecast would underperform.
- **Seasonality** — demand differs meaningfully by season and shows a weekly (day-of-week) pattern, both usable as calendar features.
- **Promotions** — promotion days show a measurable difference in average units sold, so a promotion flag/lag is a relevant feature (not proof of a causal lift, but a signal worth including).
- **Lags / previous demand** — since each SKU has its own baseline and volatility, recent-demand lag features (e.g., last 7/14/28 days) will likely help more than category-wide averages alone.
- **Volatility** — high-CV SKUs will need wider forecast intervals and more safety stock than low-CV SKUs; volatility itself is a useful feature or grouping variable for the model.

No forecasting model is built in this notebook — this section only maps today's EDA findings to what the next stage should incorporate.

## 13. Saving Useful Summary Tables

Saving reusable summary tables for later stages (forecasting, dashboarding) — not the full raw data again.

In [20]:
sku_perf_full = sku_perf.merge(vol[["SKU","Coefficient_of_Variation"]], on="SKU").merge(
    margin_by_sku[["SKU","Gross_Margin_Per_Unit"]], on="SKU")
sku_perf_full = sku_perf_full.sort_values("Revenue", ascending=False)
sku_perf_full.to_csv(PROCESSED_DIR / "sku_performance_summary.csv", index=False)

latest_snapshot.to_csv(PROCESSED_DIR / "inventory_latest_snapshot_summary.csv", index=False)

with open(REPORTS_DIR / "eda_key_findings.txt", "w") as f:
    f.write("FORESIGHT — EDA KEY FINDINGS\n")
    f.write("=" * 40 + "\n\n")
    for line in insights:
        f.write(line + "\n\n")

print("Saved:")
print(" - data/processed/sku_performance_summary.csv")
print(" - data/processed/inventory_latest_snapshot_summary.csv")
print(" - reports/eda_key_findings.txt")

Saved:
 - data/processed/sku_performance_summary.csv
 - data/processed/inventory_latest_snapshot_summary.csv
 - reports/eda_key_findings.txt
